# Sprint 1 — Automação de Coleta, Registro e Atualização
## Monitoramento Preditivo de Motores Elétricos Industriais

Execute as células em ordem. Tudo roda direto no Google Colab sem configuração adicional.

## Célula 1 — Instalação de dependências

In [ ]:
# Única dependência externa (para o dashboard formatado)
!pip install tabulate --quiet
print('✅ Dependências instaladas')

## Célula 2 — Configuração de logging e diretórios

In [ ]:
import os
import logging
import json
from pathlib import Path
from datetime import datetime

# Diretórios do projeto
BASE_DIR = Path('motor_rpa')
LOGS_DIR = BASE_DIR / 'logs'
DATA_DIR = BASE_DIR / 'data'
DB_PATH  = BASE_DIR / 'motores.db'

for d in [BASE_DIR, LOGS_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOGS_DIR / f"execucao_{datetime.now().strftime('%Y%m%d')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger('motor_rpa')
logger.info('=== Ambiente inicializado ===')
logger.info(f'Banco de dados : {DB_PATH.resolve()}')
logger.info(f'Logs           : {LOG_FILE.resolve()}')
print('✅ Ambiente configurado')
print(f'   Banco : {DB_PATH}')
print(f'   Logs  : {LOG_FILE}')

## Célula 3 — Banco de dados (SQLite + schema completo)

> **Justificativa técnica:** SQLite foi escolhido por rodar sem servidor, suportar transações ACID completas, WAL mode para concorrência e migração trivial para PostgreSQL em produção (só muda a connection string).

In [ ]:
import sqlite3
from contextlib import contextmanager

@contextmanager
def get_conn():
    """Context manager para conexão segura com rollback automático."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute('PRAGMA foreign_keys = ON')
    conn.execute('PRAGMA journal_mode = WAL')  # Write-Ahead Log: melhor concorrência
    try:
        yield conn
        conn.commit()
    except Exception as exc:
        conn.rollback()
        logger.error(f'Transação revertida: {exc}')
        raise
    finally:
        conn.close()

SCHEMA = """
CREATE TABLE IF NOT EXISTS ativos (
    id            INTEGER PRIMARY KEY AUTOINCREMENT,
    codigo        TEXT    NOT NULL UNIQUE,
    descricao     TEXT,
    localizacao   TEXT,
    potencia_kw   REAL,
    tensao_v      REAL,
    corrente_nom  REAL,
    ip_rating     TEXT,
    fabricante    TEXT,
    data_install  TEXT,
    status        TEXT    DEFAULT 'ativo',
    criado_em     TEXT    DEFAULT (datetime('now','localtime')),
    atualizado_em TEXT    DEFAULT (datetime('now','localtime'))
);

CREATE TABLE IF NOT EXISTS leituras (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    ativo_id        INTEGER NOT NULL REFERENCES ativos(id),
    fonte           TEXT    NOT NULL,
    temperatura_c   REAL,
    vibracao_mm_s   REAL,
    corrente_a      REAL,
    tensao_v        REAL,
    rpm             REAL,
    fator_potencia  REAL,
    coletado_em     TEXT    NOT NULL DEFAULT (datetime('now','localtime')),
    processado_em   TEXT,
    flag_anomalia   INTEGER DEFAULT 0,
    hash_registro   TEXT    UNIQUE
);

CREATE TABLE IF NOT EXISTS historico_atualizacoes (
    id            INTEGER PRIMARY KEY AUTOINCREMENT,
    tabela        TEXT NOT NULL,
    operacao      TEXT NOT NULL,
    registro_id   INTEGER,
    dados_antes   TEXT,
    dados_depois  TEXT,
    executor      TEXT DEFAULT 'scheduler_batch',
    executado_em  TEXT DEFAULT (datetime('now','localtime'))
);

CREATE TABLE IF NOT EXISTS log_execucoes (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    automacao       TEXT    NOT NULL,
    status          TEXT    NOT NULL,
    registros_proc  INTEGER DEFAULT 0,
    registros_erro  INTEGER DEFAULT 0,
    detalhes        TEXT,
    iniciado_em     TEXT    DEFAULT (datetime('now','localtime')),
    finalizado_em   TEXT
);

CREATE INDEX IF NOT EXISTS idx_leituras_ativo    ON leituras(ativo_id);
CREATE INDEX IF NOT EXISTS idx_leituras_coletado ON leituras(coletado_em);
CREATE INDEX IF NOT EXISTS idx_historico_tabela  ON historico_atualizacoes(tabela, registro_id);
"""

with get_conn() as conn:
    conn.executescript(SCHEMA)

logger.info('Schema criado/verificado com sucesso')
print('✅ Banco de dados pronto:', DB_PATH)

## Célula 4 — Simulador de sensores IoT

In [ ]:
import random
import hashlib
from datetime import datetime, timedelta

class SimuladorSensorIoT:
    """
    Simula um sensor IoT acoplado a um motor elétrico.
    Gera dados com ruído gaussiano e ocasionais anomalias.

    Em produção: substituir por cliente MQTT (paho-mqtt) ou OPC-UA.

    Parâmetros nominais (motor 75 kW):
      Temperatura : 50–80 °C
      Vibração    : 1–4 mm/s  (ISO 10816 Zona A/B)
      Corrente    : 80–120 A
      Tensão      : 375–415 V (±5% de 380 V)
      RPM         : 1450–1500 (4 polos 60 Hz)
      FP          : 0.82–0.92
    """

    def __init__(self, ativo_id: int, codigo: str, prob_anomalia: float = 0.04):
        self.ativo_id      = ativo_id
        self.codigo        = codigo
        self.prob_anomalia = prob_anomalia
        self._log = logging.getLogger(f'sensor.{codigo}')

    def _gerar_leitura_base(self) -> dict:
        anomalia = random.random() < self.prob_anomalia
        if anomalia:
            self._log.warning(f'[{self.codigo}] Anomalia simulada!')
            return {
                'temperatura_c' : round(random.uniform(92, 105), 2),
                'vibracao_mm_s' : round(random.uniform(7.5, 12.0), 3),
                'corrente_a'    : round(random.uniform(130, 155), 2),
                'tensao_v'      : round(random.uniform(355, 368), 1),
                'rpm'           : round(random.uniform(1390, 1430), 0),
                'fator_potencia': round(random.uniform(0.72, 0.79), 3),
                'flag_anomalia' : 1,
            }
        return {
            'temperatura_c' : round(random.gauss(65, 5), 2),
            'vibracao_mm_s' : round(random.gauss(2.5, 0.4), 3),
            'corrente_a'    : round(random.gauss(98, 8), 2),
            'tensao_v'      : round(random.gauss(390, 4), 1),
            'rpm'           : round(random.gauss(1485, 8), 0),
            'fator_potencia': round(random.gauss(0.87, 0.02), 3),
            'flag_anomalia' : 0,
        }

    def coletar(self, timestamp=None) -> dict:
        """Retorna leitura completa com metadados e hash de idempotência."""
        ts = (timestamp or datetime.now()).strftime('%Y-%m-%d %H:%M:%S')
        leitura = self._gerar_leitura_base()
        leitura.update({'ativo_id': self.ativo_id, 'fonte': 'sensor_iot', 'coletado_em': ts})
        conteudo = f"{self.ativo_id}{ts}{leitura['temperatura_c']}{leitura['corrente_a']}"
        leitura['hash_registro'] = hashlib.sha256(conteudo.encode()).hexdigest()[:32]
        return leitura

print('✅ SimuladorSensorIoT definido')

## Célula 5 — Leitor de CSV (fonte legada)

In [ ]:
import csv
import io

def gerar_csv_exemplo(ativo_id: int) -> str:
    """Gera CSV com cabeçalhos propositalmente 'sujos' (simula sistema legado)."""
    linhas = [
        ['Timestamp','Motor_ID','Temp(C)','Vibr(mm/s)','Amp','Volt','Rot(rpm)','FP'],
        ['2024-01-15 08:00','MTR-001','72.3','2.8','101.5','388','1482','0.86'],
        ['2024-01-15 08:05','MTR-001','73.1','3.1','103.2','387','1481','0.85'],
        ['2024-01-15 08:10','MTR-001','71.9','2.6','99.8','390','1484','0.87'],
        ['2024-01-15 08:15','MTR-001','95.4','8.2','142.0','362','1410','0.74'],  # anomalia
        ['2024-01-15 08:20','MTR-001','74.0','2.9','102.1','389','1483','0.86'],
    ]
    buf = io.StringIO()
    csv.writer(buf).writerows(linhas)
    return buf.getvalue()

# Mapeamento de cabeçalhos legados → campos padronizados
MAPA_COLUNAS_CSV = {
    'Temp(C)'    : 'temperatura_c',
    'Vibr(mm/s)' : 'vibracao_mm_s',
    'Amp'        : 'corrente_a',
    'Volt'       : 'tensao_v',
    'Rot(rpm)'   : 'rpm',
    'FP'         : 'fator_potencia',
    'Timestamp'  : 'coletado_em',
}

def ler_csv(conteudo_csv: str, ativo_id: int) -> list:
    """Lê CSV legado e retorna lista de dicts padronizados com hash de idempotência."""
    log_csv = logging.getLogger('leitor_csv')
    registros = []
    reader = csv.DictReader(io.StringIO(conteudo_csv))
    for i, row in enumerate(reader):
        try:
            rec = {'ativo_id': ativo_id, 'fonte': 'csv'}
            for col_legado, col_padrao in MAPA_COLUNAS_CSV.items():
                if col_legado in row:
                    rec[col_padrao] = row[col_legado]
            for campo in ['temperatura_c','vibracao_mm_s','corrente_a','tensao_v','rpm','fator_potencia']:
                if campo in rec:
                    rec[campo] = float(rec[campo])
            conteudo = f"{ativo_id}{rec.get('coletado_em','')}{rec.get('temperatura_c',0)}"
            rec['hash_registro'] = hashlib.sha256(conteudo.encode()).hexdigest()[:32]
            registros.append(rec)
        except (ValueError, KeyError) as e:
            log_csv.warning(f'Linha {i+1} ignorada: {e}')
    log_csv.info(f'{len(registros)} registros lidos do CSV')
    return registros

print('✅ Leitor de CSV definido')

## Célula 6 — Pipeline de transformação e validação

In [ ]:
from dataclasses import dataclass

@dataclass
class RegraValidacao:
    campo  : str
    minimo : float
    maximo : float
    unidade: str = ''

REGRAS_MOTOR_75KW = [
    RegraValidacao('temperatura_c',  -10,  110, '°C'),
    RegraValidacao('vibracao_mm_s',    0,   20, 'mm/s'),
    RegraValidacao('corrente_a',       0,  200, 'A'),
    RegraValidacao('tensao_v',       300,  450, 'V'),
    RegraValidacao('rpm',            800, 1800, 'rpm'),
    RegraValidacao('fator_potencia',   0,    1, ''),
]

class PipelineTransformacao:
    """
    Executa as etapas em ordem:
    1. Coerce tipos
    2. Valida limites físicos
    3. Detecta anomalias operacionais
    4. Adiciona metadados de processamento
    """
    def __init__(self, regras=None):
        self.regras = regras or REGRAS_MOTOR_75KW
        self._log   = logging.getLogger('transformer')

    def processar(self, leituras: list) -> tuple:
        """Retorna (leituras_validas, leituras_rejeitadas)."""
        validas, rejeitadas = [], []
        agora = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

        for leitura in leituras:
            erros = []

            # 1. Coerce tipos
            for regra in self.regras:
                val = leitura.get(regra.campo)
                if val is None:
                    continue
                try:
                    leitura[regra.campo] = float(val)
                except (ValueError, TypeError):
                    erros.append(f'{regra.campo}: não conversível ({val})')

            # 2. Validação de limites
            for regra in self.regras:
                val = leitura.get(regra.campo)
                if val is None:
                    continue
                if not (regra.minimo <= val <= regra.maximo):
                    erros.append(
                        f'{regra.campo}={val}{regra.unidade} '
                        f'fora de [{regra.minimo}, {regra.maximo}]'
                    )

            if erros:
                leitura['_erros_validacao'] = erros
                rejeitadas.append(leitura)
                self._log.warning(f'Leitura rejeitada: {erros}')
                continue

            # 3. Detecção de anomalia operacional
            temp = leitura.get('temperatura_c', 0)
            vibr = leitura.get('vibracao_mm_s', 0)
            corr = leitura.get('corrente_a',    0)
            if temp > 85 or vibr > 6.0 or corr > 125:
                leitura['flag_anomalia'] = 1
                self._log.warning(
                    f'⚠ Anomalia: T={temp}°C V={vibr}mm/s I={corr}A '
                    f'(ativo_id={leitura.get("ativo_id")})'
                )

            leitura['processado_em'] = agora
            validas.append(leitura)

        self._log.info(
            f'Transformação: {len(validas)} válidas, '
            f'{len(rejeitadas)} rejeitadas de {len(leituras)} total'
        )
        return validas, rejeitadas

print('✅ PipelineTransformacao definido')

## Célula 7 — Repositório: persistência com auditoria

In [ ]:
class RepositorioAtivos:
    """Camada de acesso a dados — separa lógica de negócio do SQL."""

    @staticmethod
    def inserir_ativo(dados: dict) -> int:
        sql = """
        INSERT OR IGNORE INTO ativos
            (codigo, descricao, localizacao, potencia_kw, tensao_v,
             corrente_nom, ip_rating, fabricante, data_install)
        VALUES
            (:codigo, :descricao, :localizacao, :potencia_kw, :tensao_v,
             :corrente_nom, :ip_rating, :fabricante, :data_install)
        """
        with get_conn() as conn:
            conn.execute(sql, dados)
            row = conn.execute('SELECT id FROM ativos WHERE codigo = ?', (dados['codigo'],)).fetchone()
            ativo_id = row['id']
        RepositorioAtivos._registrar_historico('ativos', 'INSERT', ativo_id, None, dados)
        logging.getLogger('repo').info(f"Ativo: {dados['codigo']} (id={ativo_id})")
        return ativo_id

    @staticmethod
    def atualizar_status(ativo_id: int, novo_status: str):
        with get_conn() as conn:
            antes = dict(conn.execute('SELECT * FROM ativos WHERE id=?', (ativo_id,)).fetchone())
            conn.execute(
                "UPDATE ativos SET status=?, atualizado_em=datetime('now','localtime') WHERE id=?",
                (novo_status, ativo_id)
            )
        RepositorioAtivos._registrar_historico('ativos', 'UPDATE', ativo_id, antes, {'status': novo_status})

    @staticmethod
    def inserir_leituras(leituras: list) -> tuple:
        """Inserção em lote com idempotência via hash_registro (INSERT OR IGNORE)."""
        sql = """
        INSERT OR IGNORE INTO leituras
            (ativo_id, fonte, temperatura_c, vibracao_mm_s, corrente_a,
             tensao_v, rpm, fator_potencia, coletado_em, processado_em,
             flag_anomalia, hash_registro)
        VALUES
            (:ativo_id, :fonte, :temperatura_c, :vibracao_mm_s, :corrente_a,
             :tensao_v, :rpm, :fator_potencia, :coletado_em, :processado_em,
             :flag_anomalia, :hash_registro)
        """
        campos = ['ativo_id','fonte','temperatura_c','vibracao_mm_s','corrente_a',
                  'tensao_v','rpm','fator_potencia','coletado_em','processado_em',
                  'flag_anomalia','hash_registro']
        registros = [{c: l.get(c) for c in campos} for l in leituras]
        with get_conn() as conn:
            antes = conn.execute('SELECT COUNT(*) FROM leituras').fetchone()[0]
            conn.executemany(sql, registros)
            depois = conn.execute('SELECT COUNT(*) FROM leituras').fetchone()[0]
        inseridos = depois - antes
        logging.getLogger('repo').info(f'Leituras: {inseridos} inseridas, {len(leituras)-inseridos} duplicatas')
        return inseridos, len(leituras) - inseridos

    @staticmethod
    def _registrar_historico(tabela, operacao, reg_id, antes, depois):
        sql = """INSERT INTO historico_atualizacoes
            (tabela, operacao, registro_id, dados_antes, dados_depois)
            VALUES (?,?,?,?,?)"""
        with get_conn() as conn:
            conn.execute(sql, (
                tabela, operacao, reg_id,
                json.dumps(antes,  default=str) if antes  else None,
                json.dumps(depois, default=str) if depois else None,
            ))

    @staticmethod
    def registrar_execucao(automacao, status, processados=0, erros=0, detalhes=None, iniciado_em=None):
        sql = """INSERT INTO log_execucoes
            (automacao, status, registros_proc, registros_erro, detalhes, iniciado_em, finalizado_em)
            VALUES (?,?,?,?,?,?,datetime('now','localtime'))"""
        with get_conn() as conn:
            conn.execute(sql, (
                automacao, status, processados, erros,
                json.dumps(detalhes or {}, default=str),
                iniciado_em or datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            ))

print('✅ RepositorioAtivos definido')

## Célula 8 — Carga inicial dos ativos

In [ ]:
ATIVOS_INICIAIS = [
    {
        'codigo'      : 'MTR-001',
        'descricao'   : 'Motor Bomba Central de Resfriamento',
        'localizacao' : 'Bloco A - Sala de Máquinas',
        'potencia_kw' : 75.0,
        'tensao_v'    : 380.0,
        'corrente_nom': 138.0,
        'ip_rating'   : 'IP55',
        'fabricante'  : 'WEG',
        'data_install': '2021-03-10',
    },
    {
        'codigo'      : 'MTR-002',
        'descricao'   : 'Motor Compressor Ar Comprimido',
        'localizacao' : 'Bloco B - Utilidades',
        'potencia_kw' : 45.0,
        'tensao_v'    : 380.0,
        'corrente_nom': 85.0,
        'ip_rating'   : 'IP54',
        'fabricante'  : 'WEG',
        'data_install': '2020-07-22',
    },
    {
        'codigo'      : 'MTR-003',
        'descricao'   : 'Motor Esteira Transportadora Linha 3',
        'localizacao' : 'Linha de Produção 3',
        'potencia_kw' : 22.0,
        'tensao_v'    : 220.0,
        'corrente_nom': 58.0,
        'ip_rating'   : 'IP65',
        'fabricante'  : 'Siemens',
        'data_install': '2022-11-05',
    },
]

ids_ativos = {}
for ativo in ATIVOS_INICIAIS:
    aid = RepositorioAtivos.inserir_ativo(ativo)
    ids_ativos[ativo['codigo']] = aid
    print(f"  → {ativo['codigo']}: id={aid}")

print(f"\n✅ {len(ATIVOS_INICIAIS)} ativos cadastrados")

## Célula 9 — Coleta histórica (7 dias)

Simula leituras a cada 5 minutos por 7 dias para popular o histórico inicial da base. Em produção esta etapa seria substituída pela carga de dados reais de SCADA/Historian.

In [ ]:
import time as time_mod

pipeline = PipelineTransformacao()
sensores = {
    codigo: SimuladorSensorIoT(aid, codigo, prob_anomalia=0.04)
    for codigo, aid in ids_ativos.items()
}

INTERVALO_MIN  = 5
DIAS_HISTORICO = 7
data_inicio    = datetime.now() - timedelta(days=DIAS_HISTORICO)
inicio_exec    = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print(f'Gerando histórico de {DIAS_HISTORICO} dias ({INTERVALO_MIN} min/leitura)...')
print(f'Data inicial: {data_inicio.strftime("%Y-%m-%d %H:%M")}')

ts, lote_total = data_inicio, []
while ts <= datetime.now():
    for codigo, sensor in sensores.items():
        lote_total.append(sensor.coletar(timestamp=ts))
    ts += timedelta(minutes=INTERVALO_MIN)

print(f'Total coletadas (raw): {len(lote_total):,}')

validas, rejeitadas = pipeline.processar(lote_total)
total_anomalias = sum(1 for l in validas if l.get('flag_anomalia') == 1)
inseridas, duplicatas = RepositorioAtivos.inserir_leituras(validas)

RepositorioAtivos.registrar_execucao(
    automacao='carga_historica_inicial', status='sucesso',
    processados=inseridas, erros=len(rejeitadas),
    detalhes={
        'periodo'   : f'{data_inicio.date()} → {datetime.now().date()}',
        'anomalias' : total_anomalias,
        'rejeitadas': len(rejeitadas),
    },
    iniciado_em=inicio_exec
)

print(f'\n{"─"*50}')
print(f'✅ Carga histórica concluída')
print(f'   Coletadas  : {len(lote_total):,}')
print(f'   Válidas    : {len(validas):,}')
print(f'   Inseridas  : {inseridas:,}')
print(f'   Anomalias  : {total_anomalias:,}')
print(f'   Rejeitadas : {len(rejeitadas):,}')

## Célula 10 — Ingestão de CSV (fonte legada)

In [ ]:
print('Ingerindo dados do CSV legado (MTR-001)...')

csv_raw  = gerar_csv_exemplo(ativo_id=ids_ativos['MTR-001'])
csv_path = DATA_DIR / 'legado_mtr001.csv'
csv_path.write_text(csv_raw, encoding='utf-8')
print(f'  CSV salvo em: {csv_path}')

registros_csv   = ler_csv(csv_raw, ativo_id=ids_ativos['MTR-001'])
validas_csv, rej_csv = pipeline.processar(registros_csv)
inseridas_csv, _ = RepositorioAtivos.inserir_leituras(validas_csv)

RepositorioAtivos.registrar_execucao(
    automacao='ingestao_csv',
    status='sucesso' if not rej_csv else 'parcial',
    processados=inseridas_csv, erros=len(rej_csv),
    detalhes={'arquivo': str(csv_path)},
)

print(f'  {inseridas_csv} registros CSV inseridos, {len(rej_csv)} rejeitados')
print('✅ Ingestão CSV concluída')

## Célula 11 — Rotina de atualização batch (scheduler)

Em produção esta função seria chamada via cron, Airflow DAG ou APScheduler. Aqui simulamos 3 ciclos completos de coleta → transformação → persistência → log.

In [ ]:
def executar_ciclo_batch(sensores: dict, n_leituras: int = 5) -> dict:
    """Um ciclo completo: Coleta → Transformação → Persistência → Auditoria."""
    log_batch = logging.getLogger('scheduler_batch')
    inicio    = datetime.now()
    log_batch.info(f"Iniciando ciclo batch | {inicio.strftime('%H:%M:%S')}")

    # 1. Coleta
    raw = []
    for codigo, sensor in sensores.items():
        ts = datetime.now()
        for _ in range(n_leituras):
            raw.append(sensor.coletar(timestamp=ts))
            ts += timedelta(seconds=30)

    # 2. Transformação
    validas, rejeitadas = pipeline.processar(raw)

    # 3. Persistência
    inseridas, ignoradas = RepositorioAtivos.inserir_leituras(validas)

    # 4. Atualizar status de motores com anomalia
    for leitura in validas:
        if leitura.get('flag_anomalia') == 1:
            RepositorioAtivos.atualizar_status(leitura['ativo_id'], 'manutencao')

    # 5. Registrar execução
    resultado = {
        'coletadas' : len(raw),
        'validas'   : len(validas),
        'inseridas' : inseridas,
        'anomalias' : sum(1 for l in validas if l.get('flag_anomalia') == 1),
        'rejeitadas': len(rejeitadas),
        'duracao_s' : round((datetime.now() - inicio).total_seconds(), 2),
    }
    RepositorioAtivos.registrar_execucao(
        automacao='batch_update', status='sucesso',
        processados=inseridas, erros=len(rejeitadas),
        detalhes=resultado,
        iniciado_em=inicio.strftime('%Y-%m-%d %H:%M:%S'),
    )
    log_batch.info(f'Ciclo concluído: {resultado}')
    return resultado

print('Executando 3 ciclos de atualização batch...\n')
for i in range(1, 4):
    res = executar_ciclo_batch(sensores, n_leituras=3)
    print(
        f"  Ciclo {i}: {res['inseridas']} inseridas | "
        f"{res['anomalias']} anomalias | {res['duracao_s']}s"
    )
    time_mod.sleep(1)

print('\n✅ Rotina batch concluída')

## Célula 12 — Dashboard de evidências

In [ ]:
try:
    from tabulate import tabulate
    FMT = 'rounded_outline'
except ImportError:
    def tabulate(rows, headers, tablefmt=None):
        print('  '.join(str(h) for h in headers))
        for r in rows: print('  '.join(str(v) for v in r))
    FMT = 'simple'

def query(sql, params=()):
    with get_conn() as conn:
        return [tuple(r) for r in conn.execute(sql, params).fetchall()]

print('\n' + '='*60)
print('DASHBOARD — MONITORAMENTO DE MOTORES ELÉTRICOS')
print('='*60)

print('\n📋 ATIVOS CADASTRADOS')
rows = query("""
    SELECT codigo, descricao, potencia_kw, status,
           (SELECT COUNT(*) FROM leituras WHERE ativo_id=ativos.id) AS leituras
    FROM ativos ORDER BY codigo
""")
print(tabulate(rows, headers=['Código','Descrição','kW','Status','Leituras'], tablefmt=FMT))

print('\n📊 RESUMO ÚLTIMAS 24H')
rows = query("""
    SELECT a.codigo,
           COUNT(l.id)                   AS total,
           ROUND(AVG(l.temperatura_c),1) AS temp_media,
           ROUND(MAX(l.temperatura_c),1) AS temp_max,
           ROUND(AVG(l.vibracao_mm_s),2) AS vibr_media,
           ROUND(AVG(l.corrente_a),1)    AS amp_media,
           SUM(l.flag_anomalia)          AS anomalias
    FROM leituras l JOIN ativos a ON a.id = l.ativo_id
    WHERE l.coletado_em >= datetime('now','-1 day','localtime')
    GROUP BY a.codigo ORDER BY a.codigo
""")
print(tabulate(rows, headers=['Motor','Leituras','Temp Média','Temp Max','Vibr','Amp','Anomalias'], tablefmt=FMT))

print('\n⚠  ÚLTIMAS 5 ANOMALIAS')
rows = query("""
    SELECT a.codigo, l.temperatura_c, l.vibracao_mm_s, l.corrente_a, l.coletado_em, l.fonte
    FROM leituras l JOIN ativos a ON a.id = l.ativo_id
    WHERE l.flag_anomalia = 1 ORDER BY l.coletado_em DESC LIMIT 5
""")
print(tabulate(rows, headers=['Motor','Temp °C','Vibr mm/s','Amp','Timestamp','Fonte'], tablefmt=FMT))

print('\n🤖 LOG DE EXECUÇÕES DAS AUTOMAÇÕES')
rows = query("""
    SELECT automacao, status, registros_proc, registros_erro, iniciado_em, finalizado_em
    FROM log_execucoes ORDER BY id DESC LIMIT 8
""")
print(tabulate(rows, headers=['Automação','Status','Inseridos','Erros','Iniciado','Finalizado'], tablefmt=FMT))

print('\n📁 HISTÓRICO DE AUDITORIA (últimas 5 entradas)')
rows = query("""
    SELECT tabela, operacao, registro_id, executado_em
    FROM historico_atualizacoes ORDER BY id DESC LIMIT 5
""")
print(tabulate(rows, headers=['Tabela','Operação','ID Registro','Executado em'], tablefmt=FMT))

tamanho_kb     = round(DB_PATH.stat().st_size / 1024, 1)
total_leituras = query('SELECT COUNT(*) FROM leituras')[0][0]
print(f'\n💾 Banco: {tamanho_kb} KB  |  Total de leituras: {total_leituras:,}')

## Célula 13 — Exportar evidências para download

In [ ]:
import zipfile

# Exibir últimas linhas do log
print('=== Últimas 15 linhas do log de execução ===\n')
for linha in LOG_FILE.read_text(encoding='utf-8').splitlines()[-15:]:
    print(linha)

# Compactar evidências
zip_path = BASE_DIR / 'evidencias_sprint1.zip'
with zipfile.ZipFile(zip_path, 'w') as z:
    for arq in [DB_PATH, LOG_FILE, csv_path]:
        if arq.exists():
            z.write(arq, arq.name)
            print(f'\n📦 {arq.name} adicionado ao zip')

print(f'\n✅ Evidências em: {zip_path.resolve()}')
print('\n' + '='*60)
print('SPRINT 1 CONCLUÍDA — TODOS OS ENTREGÁVEIS GERADOS')
print('='*60)

# Descomente para download automático no Colab:
# from google.colab import files
# files.download(str(zip_path))